In [1]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import math

In [9]:
# Define buffer for robot size (increased to 0.3m for better margins)
buffer = 0.3

# List of obstacles as [min_x, min_y, max_x, max_y] with refined boundaries
obstacles = [
    # Main walls
    [-6.443, -3.86, -6.383, 3.86],  # West wall
    [6.383, -3.86, 6.443, 3.86],  # East wall
    [-6.4, -3.86, 6.4, -3.8],  # South wall approximation
    [-6.4, 3.8, 6.4, 3.86],  # North wall approximation

    # Internal small walls
    [-2.918, -3.43, -2.868, 1.57],  # wall
    [2.382, -3.475, 2.432, 0.075],  # wall(1)
    [-2.918, -3.475, 2.432, -3.425],  # wall(2)
    [-2.918, 1.569, -0.868, 1.619],  # wall(3)
    [-0.918, 1.62, -0.868, 2.62],  # wall(4)
    [0.132, 1.62, 0.182, 2.62],  # wall(5)
    [0.133, 1.569, 3.133, 1.619],  # wall(6)
    [2.433, 0.024, 3.133, 0.074],  # wall(7)

    # Sofa
    [0.087, -3.33, 1.887, -2.53],

    # Potted Trees (assume 0.4x0.4 footprint)
    [1.937, -3.28, 2.337, -2.88],  # potted tree
    [5.787, 0.29, 6.187, 0.69],  # potted tree(1)
    [-6.223, -3.3, -5.823, -2.9],  # potted tree(2)

    # Cabinets (approximate footprints, assume depth 0.44m)
    [-0.473, -3.74, -0.033, -3.1],  # cabinet
    [1.857, -0.93, 2.897, -0.49],  # cabinet(1)
    # [4.787, -1.85, 8.027, -1.41],  # cabinet(2) - note: may clip wall
    [0.95, 1.35, 1.57, 1.79],  # cabinet(4) rotation -1.57, approximate
    [-1.123, -3.74, -0.683, -3.1],  # cabinet(5)
    [-2.063, -3.63, -1.463, -3.23],  # cabinet(6) assume 1x0.4 after rotation
    [-2.753, -3.63, -2.353, -3.23],  # cabinet(7)

    # Sink (assume 0.6x0.5 footprint)
    [-2.813, -3.4, -2.213, -3.1],

    # Tables (use specified sizes)
    [2.437, -3.39, 3.437, -1.79],  # table size 1x1.6
    [5.397, -3.39, 6.397, -1.79],  # table(1) size 1x1.6
    [-3.923, -3.39, -2.923, -1.79],  # table(2) size 1x1.6
    [2.437, -1.28, 3.437, -0.48],  # table(3) default 1x0.8
    [5.397, -1.28, 6.397, -0.48],  # table(4)
    [-3.923, -1.28, -2.923, -0.48],  # table(5)
    [-3.923, 0.53, -2.923, 1.33],  # table(6)
    [-2.413, 1.84, -1.613, 2.84],  # table(7) rotated, size 1x0.8 -> 0.8x1
    [-3.923, 1.84, -2.923, 2.84],  # table(8) rotated, size 1x1

    # Office Chairs (assume 0.6x0.6 footprint)
    [3.575, -1.175, 3.975, -0.775],  # office chair
    [3.497, -2.66, 3.897, -2.26],  # office chair(1)
    [4.921, -2.613, 5.321, -2.213],  # office chair(2)
    [-4.612, -3.111, -4.212, -2.711],  # office chair(3)
    [-4.443, -1.06, -4.043, -0.66],  # office chair(4)
    [-4.513, 0.72, -4.113, 1.12],  # office chair(5)

]

# Expand obstacles by buffer
expanded_obstacles = []
for obs in obstacles:
    expanded_obstacles.append([
        obs[0] - buffer,
        obs[1] - buffer,
        obs[2] + buffer,
        obs[3] + buffer
    ])

# Function to check if point is free
def is_free(point, expanded_obstacles):
    x, y = point
    for o in expanded_obstacles:
        if o[0] <= x <= o[2] and o[1] <= y <= o[3]:
            return False
    return True

# Generate dense grid
spacing = 0.035  # Dense spacing
x_vals = np.arange(-6.3, 6.3 + spacing, spacing)
y_vals = np.arange(-3.7, 3.7 + spacing, spacing)
xx, yy = np.meshgrid(x_vals, y_vals)
all_points = np.c_[xx.ravel(), yy.ravel()]

# Filter free points
free_points = [p for p in all_points if is_free(p, expanded_obstacles)]

# Convert to numpy array
free_points = np.array(free_points)

In [10]:
# Generate XML content
xml_content = '<?xml version="1.0" encoding="us-ascii"?>\n'
xml_content += '<world>\n'
xml_content += '    <experimentStartPositions>\n'
for point in free_points:
    x, y = point
    xml_content += f'        <pos x="{x:.2f}" y="{y:.2f}" theta="1.5707963267948966" />\n'
xml_content += '    </experimentStartPositions>\n'
xml_content += '    <habituationStartPositions>\n'
xml_content += '        <pos x="0.00" y="0.0" theta="1.5707963267948966" />\n'
xml_content += '    </habituationStartPositions>\n'
xml_content += '    <goal id="1" x="0.0" y="2.5" />\n'
xml_content += '</world>'

# Save to XML file
with open('start_positions_BR.xml', 'w') as f:
    f.write(xml_content)
# Load the image and plot points superimposed
img_path = '../../../data/DataCache/BR.png'
img = Image.open(img_path)

# Define extent based on room dimensions (adjust if image has padding)
x_min, x_max = -6.4, 6.4
y_min, y_max = -3.85, 3.85

fig, ax = plt.subplots(figsize=(12, 7))  # Adjust figsize to match aspect ratio if needed
ax.imshow(img, extent=[x_min, x_max, y_min, y_max])
ax.scatter(free_points[:, 0], free_points[:, 1], s=1, color='red', alpha=0.5)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Free Points Superimposed on Break Room Image')

# Save the plot
plt.savefig('points_on_map_BR.png')
plt.close()

print(f"Generated {len(free_points)} free points.")
print("Points saved to 'start_positions.xml'.")
print("Plot saved to 'points_on_map.png'.")

Generated 39645 free points.
Points saved to 'start_positions.xml'.
Plot saved to 'points_on_map.png'.


In [2]:
# Define buffer for robot size (0.3m)
buffer = 0.3

# Define the octagon vertices in clockwise order
vertices = [
    (2.61, 0.00),
    (1.85, 1.85),
    (0.00, 2.61),
    (-1.85, 1.85),
    (-2.61, 0.00),
    (-1.85, -1.85),
    (0.00, -2.61),
    (1.85, -1.85)
]

# Define the wall segments (connecting consecutive vertices, closing the loop)
segments = list(zip(vertices, vertices[1:] + [vertices[0]]))

# Function to check if a point is inside the polygon (ray casting algorithm)
def point_in_polygon(x, y, poly):
    n = len(poly)
    inside = False
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Function to compute distance from point to line segment
def dist_to_segment(p, seg_start, seg_end):
    px, py = p
    ax, ay = seg_start
    bx, by = seg_end
    abx = bx - ax
    aby = by - ay
    apx = px - ax
    apy = py - ay
    proj = apx * abx + apy * aby
    len_sq = abx**2 + aby**2
    if len_sq == 0:
        return np.sqrt(apx**2 + apy**2)
    t = max(0, min(1, proj / len_sq))
    qx = ax + t * abx
    qy = ay + t * aby
    dx = px - qx
    dy = py - qy
    return np.sqrt(dx**2 + dy**2)

# Function to check if point is free (inside polygon and away from walls)
def is_free(point, segments, buffer):
    x, y = point
    if not point_in_polygon(x, y, vertices):
        return False
    for seg in segments:
        if dist_to_segment((x, y), seg[0], seg[1]) < buffer:
            return False
    return True

# Generate dense grid
spacing = 0.05  # Dense spacing
x_vals = np.arange(-3.0, 3.0 + spacing, spacing)
y_vals = np.arange(-3.0, 3.0 + spacing, spacing)
xx, yy = np.meshgrid(x_vals, y_vals)
all_points = np.c_[xx.ravel(), yy.ravel()]

# Filter free points
free_points = [p for p in all_points if is_free(p, segments, buffer)]

# Convert to numpy array
free_points = np.array(free_points)

In [5]:
# Generate XML content
xml_content = '<?xml version="1.0" encoding="us-ascii"?>\n'
xml_content += '<world>\n'
xml_content += '    <experimentStartPositions>\n'
for point in free_points:
    x, y = point
    xml_content += f'        <pos x="{x:.2f}" y="{y:.2f}" theta="1.5707963267948966" />\n'
xml_content += '    </experimentStartPositions>\n'
xml_content += '    <habituationStartPositions>\n'
xml_content += '        <pos x="0.00" y="0.0" theta="1.5707963267948966" />\n'
xml_content += '    </habituationStartPositions>\n'
xml_content += '    <goal id="1" x="0.0" y="2.5" />\n'
xml_content += '</world>'

# Save to XML file
with open('start_positions_LM8.xml', 'w') as f:
    f.write(xml_content)
# Load the image and plot points superimposed
img_path = '../../../data/DataCache/LM8.png'
img = Image.open(img_path)

# Define extent based on room dimensions (adjust if image has padding)
x_min, x_max = -3, 3
y_min, y_max = -3, 3

fig, ax = plt.subplots(figsize=(12, 7))  # Adjust figsize to match aspect ratio if needed
ax.imshow(img, extent=[x_min, x_max, y_min, y_max])
ax.scatter(free_points[:, 0], free_points[:, 1], s=1, color='red', alpha=0.5)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Free Points Superimposed on Break Room Image')

# Save the plot
plt.savefig('points_on_map_LM8.png')
plt.close()

print(f"Generated {len(free_points)} free points.")
print("Points saved to 'start_positions_LM8.xml'.")
print("Plot saved to 'points_on_map_LM8.png'.")

Generated 23701 free points.
Points saved to 'start_positions_LM8.xml'.
Plot saved to 'points_on_map_LM8.png'.


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ---- Parameters (same style as your original) ----
buffer = 0.2
spacing = 0.04
theta = 1.5707963267948966

# Keep your original vertices list exactly as before
vertices = [
    (2.61, 0.00),
    (1.85, 1.85),
    (0.00, 2.61),
    (-1.85, 1.85),
    (-2.61, 0.00),
    (-1.85, -1.85),
    (0.00, -2.61),
    (1.85, -1.85),
]

# Build the true outer boundary segments from vertices (same as your first script)
boundary_segments = list(zip(vertices, vertices[1:] + [vertices[0]]))

# Only these are internal walls — keep them
internal_walls = [
    (( 0.75, -1.25), (-0.75, -1.25)),
    (( 0.75,  1.25), (-0.75,  1.25)),
    (( 1.25,  0.75), ( 1.25, -0.75)),
    ((-1.25,  0.75), (-1.25, -0.75)),
    (( 0.50, -0.50), (-0.50,  0.50)),
    ((-0.50, -0.50), ( 0.50,  0.50)),
]

# Use boundary + internal for clearance checks
segments = boundary_segments + internal_walls


# ---- Same helpers as your original ----
def point_in_polygon(x, y, poly):
    n = len(poly)
    inside = False
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

def dist_to_segment(p, seg_start, seg_end):
    px, py = p
    ax, ay = seg_start
    bx, by = seg_end
    abx = bx - ax
    aby = by - ay
    apx = px - ax
    apy = py - ay
    proj = apx * abx + apy * aby
    len_sq = abx**2 + aby**2
    if len_sq == 0:
        return np.sqrt(apx**2 + apy**2)
    t = max(0, min(1, proj / len_sq))
    qx = ax + t * abx
    qy = ay + t * aby
    dx = px - qx
    dy = py - qy
    return np.sqrt(dx**2 + dy**2)

def is_free(point, segments, buffer):
    # Uses the GLOBAL 'vertices' for inside-test (exactly like your original)
    x, y = point
    if not point_in_polygon(x, y, vertices):
        return False
    for seg in segments:
        if dist_to_segment((x, y), seg[0], seg[1]) < buffer:
            return False
    return True

# ---- Dense grid ----
x_vals = np.arange(-3.0, 3.0 + spacing, spacing)
y_vals = np.arange(-3.0, 3.0 + spacing, spacing)
xx, yy = np.meshgrid(x_vals, y_vals)
all_points = np.c_[xx.ravel(), yy.ravel()]

# ---- Filter free points (now includes buffer to true outer boundary) ----
free_points = [p for p in all_points if is_free(p, segments, buffer)]
free_points = np.array(free_points)

# ---- Write XML (same format as before) ----
xml_content = '<?xml version="1.0" encoding="us-ascii"?>\n'
xml_content += '<world>\n'
xml_content += '    <experimentStartPositions>\n'
for x, y in free_points:
    xml_content += f'        <pos x="{x:.2f}" y="{y:.2f}" theta="{theta}" />\n'
xml_content += '    </experimentStartPositions>\n'
xml_content += '    <habituationStartPositions>\n'
xml_content += f'        <pos x="0.00" y="0.00" theta="{theta}" />\n'
xml_content += '    </habituationStartPositions>\n'
xml_content += '    <goal id="1" x="0.00" y="2.50" />\n'
xml_content += '</world>'

with open('start_positions_LM8.xml', 'w') as f:
    f.write(xml_content)

# ---- Plot (unchanged) ----
img_path = '../../../data/DataCache/LMO8.png'
img = Image.open(img_path)

x_min, x_max = -3, 3
y_min, y_max = -3, 3

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img, extent=[x_min, x_max, y_min, y_max])
if free_points.size > 0:
    ax.scatter(free_points[:, 0], free_points[:, 1], s=1, color='red', alpha=0.5)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Free Points Superimposed on Break Room Image')
ax.set_aspect('equal', adjustable='box')

plt.savefig('points_on_map_LM8.png', dpi=200)
plt.close()

print(f"Generated {len(free_points)} free points.")
print("Points saved to 'start_positions_LM8.xml'.")
print("Plot saved to 'points_on_map_LM8.png'.")


Generated 7544 free points.
Points saved to 'start_positions_LM8.xml'.
Plot saved to 'points_on_map_LM8.png'.


In [ ]:
from typing import List, Tuple
# ---- Internal walls from your XML ----

WALL_DEFS: List[Tuple[Tuple[float, float], Tuple[float, float]]] = [
    ((0.75,  -1.25), (-0.75, -1.25)),  # Wall 1 (bottom horizontal)
    ((0.75,   1.25), (-0.75,  1.25)),  # Wall 2 (top horizontal)
    ((1.25,   0.75), ( 1.25, -0.75)),  # Wall 3 (right vertical)
    ((-1.25,  0.75), (-1.25, -0.75)),  # Wall 4 (left vertical)
    ((0.50,  -0.50), (-0.50,  0.50)),  # Wall 5 (diagonal)
    ((-0.50, -0.50), ( 0.50,  0.50)),  # Wall 6 (diagonal)
]

In [ ]:
# ============================================================
# WALL DEFINITIONS
# ============================================================

# Internal box walls
BOX_WALLS = [
    ((0.75, -1.25), (-0.75, -1.25)),  # W1 bottom
    ((0.75,  1.25), (-0.75,  1.25)),  # W2 top
    ((1.25,  0.75), ( 1.25, -0.75)),  # W3 right
    ((-1.25, 0.75), (-1.25, -0.75)),  # W4 left
]

# Internal X walls
X_WALLS = [
    ((0.50, -0.50), (-0.50,  0.50)),  # W5
    ((-0.50,-0.50), ( 0.50,  0.50)),  # W6
]

# Outer octagon walls
OUTER_WALLS = [
    ((2.61, 1.85), (1.85, 0.00)),
    ((1.85, 2.61), (0.00, 1.85)),
    ((0.00, 1.85), (-1.85, 2.61)),
    ((-1.85, 0.00), (-2.61, 1.85)),
    ((-2.61, 0.00), (-1.85, -1.85)),
    ((-1.85, -1.85), (-0.00, -2.61)),
    ((-0.00, -2.61), (1.85, -1.85)),
    ((1.85, -1.85), (2.61, 0.00)),
]

# Pocket points around X
POCKET_POINTS = [
    (0.0,  0.5),
    (0.0, -0.5),
    (-0.5, 0.0),
    (0.5,  0.0),
]

# ============================================================
# POINT SAMPLING
# ============================================================

def sample_wall_points(p1, p2, offset=0.25, n_samples=3):
    """
    Sample points along a wall and return inside/outside offset points.
    Returns list of (base, plus_offset, minus_offset).
    """
    p1 = np.array(p1, float)
    p2 = np.array(p2, float)
    v = p2 - p1

    # Normal vector
    n = np.array([-v[1], v[0]], dtype=float)
    n = n / np.linalg.norm(n)

    ts = np.linspace(0.25, 0.75, n_samples)
    out = []
    for t in ts:
        base = p1 + t * v
        out.append((base, base + offset * n, base - offset * n))
    return out


def generate_box_wall_test_points(offset=0.25):
    pts = {f"W{i}_inside": [] for i in range(1, 5)}
    pts.update({f"W{i}_outside": [] for i in range(1, 5)})

    # W1 bottom
    for base, plus, minus in sample_wall_points(*BOX_WALLS[0], offset):
        inside = plus if plus[1] > base[1] else minus
        outside = minus if inside is plus else plus
        pts["W1_inside"].append(tuple(inside))
        pts["W1_outside"].append(tuple(outside))

    # W2 top
    for base, plus, minus in sample_wall_points(*BOX_WALLS[1], offset):
        inside = minus if minus[1] < base[1] else plus
        outside = plus if inside is minus else minus
        pts["W2_inside"].append(tuple(inside))
        pts["W2_outside"].append(tuple(outside))

    # W3 right
    for base, plus, minus in sample_wall_points(*BOX_WALLS[2], offset):
        inside = minus if minus[0] < base[0] else plus
        outside = plus if inside is minus else minus
        pts["W3_inside"].append(tuple(inside))
        pts["W3_outside"].append(tuple(outside))

    # W4 left
    for base, plus, minus in sample_wall_points(*BOX_WALLS[3], offset):
        inside = plus if plus[0] > base[0] else minus
        outside = minus if inside is plus else plus
        pts["W4_inside"].append(tuple(inside))
        pts["W4_outside"].append(tuple(outside))

    return pts

# ============================================================
# PLOTTING
# ============================================================

def plot_maze(points_dict):
    plt.figure(figsize=(7, 7))

    # Outer walls
    for p1, p2 in OUTER_WALLS:
        plt.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k-', linewidth=2)

    # Internal box walls
    for p1, p2 in BOX_WALLS:
        plt.plot([p1[0], p2[0]], [p1[1], p2[1]], 'b-', linewidth=2)

    # Internal X walls
    for p1, p2 in X_WALLS:
        plt.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=2)

    # Test points
    for label, pts in points_dict.items():
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        plt.scatter(xs, ys, s=80, label=label)

    # Pockets
    px = [p[0] for p in POCKET_POINTS]
    py = [p[1] for p in POCKET_POINTS]
    plt.scatter(px, py, s=120, c='magenta', marker='X', label='Pockets')

    plt.legend(loc='upper right', fontsize=8)
    plt.gca().set_aspect('equal', 'box')
    plt.title("Maze Walls and Sampling Points")
    plt.grid(True)
    plt.show()

# ============================================================
# XML EXPORT
# ============================================================

def export_points_to_xml(points_dict, pockets=POCKET_POINTS,
                         theta=1.5707963267948966,
                         filename="test_points.xml"):

    with open(filename, "w") as f:
        f.write("<positions>\n")

        # Box wall points
        for label, pts in points_dict.items():
            for x, y in pts:
                f.write(f'  <pos x="{x:.3f}" y="{y:.3f}" theta="{theta}" />\n')

        # Pockets
        for x, y in pockets:
            f.write(f'  <pos x="{x:.3f}" y="{y:.3f}" theta="{theta}" />\n')

        f.write("</positions>\n")

    print(f"XML exported → {filename}")

# ============================================================
# RUN EVERYTHING (Jupyter-friendly)
# ============================================================

points = generate_box_wall_test_points(offset=0.3)
plot_maze(points)
export_points_to_xml(points)
